# 06 - Evaluation
**Stage:** match each model answer to the ground-truth face and compute the metrics.

**Matching rule.** For numbered variants (V4, V5, V6, V8) the answer for face number *k* is compared with ground-truth face *k*.
For the other variants the model was told to list faces left to right, so answer *k* is compared with ground-truth face *k*
(faces are numbered left to right). A face with no answer, an invalid label or an unparsable answer counts as wrong.

**Metrics.** Accuracy = correct faces / all faces. Majority-class baseline = accuracy of always answering the most common true emotion.
Recall / precision / F1 = macro over the emotions present in the ground truth. Face-count accuracy = share of images where the number of
faces is exactly right. Differences vs. the baseline get a 95% bootstrap interval (resampling images).

| | |
|---|---|
| **Input** | `data/dataset.json` (or `data/dataset_eval_only.json`, no photos needed), `results/<model>/raw_outputs.jsonl` |
| **Output** | `metrics.csv`, `per_face_predictions.csv`, `bootstrap_vs_baseline.csv`, `per_class_recall.csv`, `results_summary.md` |
| **Code used** | `src/evaluation.py` |

This notebook runs on a laptop CPU - no GPU and no photos needed (it uses the saved answers).

In [1]:
# Setup: find the repo root so `from src import ...` works (run this notebook from inside the repo).
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").is_dir()), None)
assert ROOT is not None, "Run from inside the repo (the folder containing src/). In Colab: upload the repo zip, unzip it and %cd into it."
sys.path.insert(0, str(ROOT))
from src import config, utils
print("repo root:", ROOT)

repo root: /home/claude/repo/sov-emotion-recognition


In [2]:
import pandas as pd
from src import evaluation, labeling
PROVIDER = "qwen"                 # whose answers to evaluate: qwen | llava | gemini
VARIANTS = config.VARIANTS

DATA = utils.load_dataset()
RES = config.results_dir(PROVIDER)
RAW = utils.load_raw_outputs(RES / "raw_outputs.jsonl")
res = evaluation.evaluate(DATA, RAW, VARIANTS)
FACES, IMGS, M, CI = res["faces"], res["imgs"], res["M"], res["ci"]
print(f"evaluating {len(res['eval_data'])} of {len(DATA)} images | answer status:", res["status"].to_dict(), "(failed answers count as wrong)")

evaluating 17 of 17 images | answer status: {'ok': 136} (failed answers count as wrong)


In [3]:
pd.options.display.float_format = "{:.3f}".format
for col, title in [("accuracy", "Accuracy (per face)"), ("recall", "Recall (macro)"), ("precision", "Precision (macro)"),
                   ("f1", "F1 (macro)"), ("face_count", "Face-count accuracy (per image)")]:
    print("\n==", title)
    print(evaluation.wide(M, col, VARIANTS).to_string(index=False))
print("\n== Images / faces per difficulty")
print(M[M.variant == VARIANTS[0]][["difficulty", "images", "faces"]].to_string(index=False))


== Accuracy (per face)
                              Variant  Easy  Medium  Hard   All
                  V0 Plain (baseline) 0.455   0.526 0.111 0.468
   V1 RedCircle-style (circle + blur) 0.727   0.561 0.111 0.532
         V2 ReCLIP-style (box + blur) 0.273   0.526 0.000 0.429
                        V3 Boxes only 0.364   0.491 0.111 0.429
                   V4 Boxes + numbers 0.545   0.526 0.222 0.494
V5 Full SoV (boxes+numbers+landmarks) 0.364   0.456 0.222 0.416
                      V6 Numbers only 0.545   0.544 0.000 0.481
                    V7 Landmarks only 0.273   0.404 0.111 0.351

== Recall (macro)
                              Variant  Easy  Medium  Hard   All
                  V0 Plain (baseline) 0.360   0.292 0.125 0.287
   V1 RedCircle-style (circle + blur) 0.667   0.315 0.125 0.337
         V2 ReCLIP-style (box + blur) 0.280   0.292 0.000 0.270
                        V3 Boxes only 0.480   0.229 0.125 0.270
                   V4 Boxes + numbers 0.560   0.298 0.250 0.3

In [4]:
print("== Accuracy difference vs V0 plain, 95% bootstrap CI over images")
print(CI.assign(variant=CI.variant.map(config.VARIANT_TITLES)).to_string(index=False))
print("\n== Per-class recall (share of true faces of that emotion predicted correctly)")
print(res["per_class"].to_string())
print(f"\n== Majority-class baseline: always answering '{res['maj_class']}' gives accuracy {res['maj']:.1%}")
for v, top, share in res["top_answers"]:
    flag = "   <-- COLLAPSE: one answer for almost every face, accuracy is not meaningful" if v in res["collapsed"] else ""
    print(f"{config.VARIANT_TITLES[v]:45s} most common answer: {top} ({share:.0%}){flag}")

== Accuracy difference vs V0 plain, 95% bootstrap CI over images
                              variant  acc_diff  ci_low  ci_high
   V1 RedCircle-style (circle + blur)     0.065   0.011    0.129
         V2 ReCLIP-style (box + blur)    -0.039  -0.096    0.014
                        V3 Boxes only    -0.039  -0.097    0.013
                   V4 Boxes + numbers     0.026  -0.043    0.095
V5 Full SoV (boxes+numbers+landmarks)    -0.052  -0.193    0.060
                      V6 Numbers only     0.013  -0.079    0.110
                    V7 Landmarks only    -0.117  -0.260   -0.011

== Per-class recall (share of true faces of that emotion predicted correctly)
truth              angry  disgust  fear  happy  neutral   sad  surprise
variant                                                                
V0_plain           0.167    0.000 0.000  0.750    0.593 0.000     0.500
V1_redcircle       0.333    0.000 0.000  0.792    0.667 0.067     0.500
V2_blur_outside    0.167    0.000 0.000  0.667  

In [5]:
M.to_csv(RES / "metrics.csv", index=False)
FACES.to_csv(RES / "per_face_predictions.csv", index=False)
CI.to_csv(RES / "bootstrap_vs_baseline.csv", index=False)
res["per_class"].to_csv(RES / "per_class_recall.csv")
per_diff, emo = labeling.dataset_stats(res["eval_data"])
(RES / "dataset_stats.md").write_text(f"Images {len(res['eval_data'])}, faces {int(per_diff.faces.sum())}\n\n" + per_diff.to_string() + "\n\n" + emo.to_string())
(RES / "results_summary.md").write_text(evaluation.summary_markdown(res, VARIANTS, RES.name))
print("saved metrics and summary to", RES)

saved metrics and summary to /home/claude/repo/sov-emotion-recognition/results/Qwen_Qwen2-VL-7B-Instruct


**Next:** `07_analysis_and_report.ipynb`